# AI Engineer Assessment Chatbot

## 1. Setup and Dependencies

This project uses:
- Groq for hosted LLM inference
- Superhero API for superhero information
- TF-IDF for text retrieval
- FastAPI for the `/ask` endpoint

### API Configuration

Create a `.env` file in the project root:

GROQ_API_KEY=your_key

SUPERHERO_API_TOKEN=your_token



## 1.1 Install dependencies


In [50]:
!pip install -q -r requirements.txt


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1.2 Imports

In [51]:
import json
import requests
from enum import Enum
from typing import Optional

from pydantic import BaseModel, Field
from groq import Groq

## 2. API Keys and Configuration

### 2.1 API Keys


In [52]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
SUPERHERO_API_TOKEN = os.getenv("SUPERHERO_API_TOKEN")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is not set.")

if not SUPERHERO_API_TOKEN:
    raise ValueError("SUPERHERO_API_TOKEN is not set.")

print("Groq API key loaded:", GROQ_API_KEY is not None)
print("Superhero API token loaded:", SUPERHERO_API_TOKEN is not None)

Groq API key loaded: True
Superhero API token loaded: True


In [53]:
client = Groq(api_key=GROQ_API_KEY)

## 3. Information Sources


### 3.1 Superhero API

In [105]:

SUPERHERO_API_BASE_URL = "https://superheroapi.com/api"


def search_superhero(name: str) -> dict:
    """
    Search for a superhero using the Superhero API.
    """

    url = f"{SUPERHERO_API_BASE_URL}/{SUPERHERO_API_TOKEN}/search/{name}"

    try:
        response = requests.get(
            url,
            timeout=30
        )

        response.raise_for_status()

        return response.json()

    except requests.RequestException as e:
        raise RuntimeError(
            f"Superhero API request failed: {str(e)}"
        )

### Test the Superhero API

In [55]:
result = search_superhero("Batman")

print("API response:", result.get("response"))
print("Number of results:", len(result.get("results", [])))

API response: success
Number of results: 3


### 3.2 Text Dataset

In [56]:
text_documents = {
    "rag.txt": """
Retrieval-Augmented Generation (RAG) is a technique that combines
information retrieval with text generation. Instead of relying only
on the knowledge stored in a language model, RAG retrieves relevant
documents from an external knowledge source and provides them to the
language model as context.

A typical RAG system has two main stages. First, a retriever searches
a collection of documents and finds the most relevant passages.
Second, a language model uses the retrieved passages to generate
an answer.

RAG is useful when the information comes from a private, changing,
or domain-specific knowledge base.
""",

    "llm.txt": """
Large Language Models (LLMs) are machine learning models trained on
large collections of text. They can perform tasks such as text
generation, summarization, question answering, and information
extraction.

LLMs generate responses based on patterns learned during training.
They can also be provided with external context at inference time,
for example through Retrieval-Augmented Generation.
""",

    "machine_learning.txt": """
Machine learning is a branch of artificial intelligence in which
models learn patterns from data.

Supervised learning uses labeled examples to train a model.
Unsupervised learning works with data without explicit labels.
Common supervised learning tasks include classification and
regression.

A machine learning pipeline commonly includes data preparation,
feature processing, model training, validation, and evaluation.
"""
}

print("Number of documents:", len(text_documents))

for name in text_documents:
    print("-", name)

Number of documents: 3
- rag.txt
- llm.txt
- machine_learning.txt


## 4. Text Processing and Retrieval


### 4.1 Chunking

In [57]:
def chunk_text(text: str, chunk_size: int = 500):
    """
    Split text into smaller chunks.

    Each chunk will later be searchable by the retriever.
    """
    
    words = text.split()
    
    chunks = []
    
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    
    return chunks


text_chunks = []

for document_name, document_text in text_documents.items():
    
    chunks = chunk_text(document_text)
    
    for chunk in chunks:
        text_chunks.append({
            "document": document_name,
            "content": chunk
        })


print("Number of chunks:", len(text_chunks))

for chunk in text_chunks:
    print("\nDocument:", chunk["document"])
    print(chunk["content"][:200], "...")

Number of chunks: 3

Document: rag.txt
Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text generation. Instead of relying only on the knowledge stored in a language model, RAG retrieves relevan ...

Document: llm.txt
Large Language Models (LLMs) are machine learning models trained on large collections of text. They can perform tasks such as text generation, summarization, question answering, and information extrac ...

Document: machine_learning.txt
Machine learning is a branch of artificial intelligence in which models learn patterns from data. Supervised learning uses labeled examples to train a model. Unsupervised learning works with data with ...


### 4.2 TF-IDF index

In [58]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [59]:
chunk_contents = [
    chunk["content"]
    for chunk in text_chunks
]

vectorizer = TfidfVectorizer(
    stop_words="english"
)

text_matrix = vectorizer.fit_transform(chunk_contents)

print("Number of chunks:", len(chunk_contents))
print("TF-IDF matrix shape:", text_matrix.shape)

Number of chunks: 3
TF-IDF matrix shape: (3, 88)


### 4.3 Text Retriever

In [60]:
def retrieve_text(
    question: str,
    top_k: int = 3,
    threshold: float = 0.1
):
    """
    Retrieve the most relevant text chunks for a question.
    """

    # Convert the question into the same TF-IDF space
    query_vector = vectorizer.transform([question])

    # Compare the question with every text chunk
    similarity_scores = cosine_similarity(
        query_vector,
        text_matrix
    )[0]

    # Rank chunks from most relevant to least relevant
    ranked_indices = similarity_scores.argsort()[::-1]

    results = []

    for index in ranked_indices[:top_k]:

        score = float(similarity_scores[index])

        # Only keep sufficiently relevant chunks
        if score >= threshold:

            results.append({
                "document": text_chunks[index]["document"],
                "content": text_chunks[index]["content"],
                "score": score
            })

    return results

### Testing retriever

In [61]:
test_questions = [
    "What is retrieval augmented generation?",
    "What are large language models?",
    "What is supervised learning?",
    "Who is Batman?"
]

for question in test_questions:

    results = retrieve_text(question)

    print("\nQuestion:", question)

    if not results:
        print("No relevant text found.")
        continue

    for result in results:
        print(
            f"Document: {result['document']} | "
            f"Score: {result['score']:.3f}"
        )


Question: What is retrieval augmented generation?
Document: llm.txt | Score: 0.293
Document: rag.txt | Score: 0.241

Question: What are large language models?
Document: llm.txt | Score: 0.425
Document: rag.txt | Score: 0.130

Question: What is supervised learning?
Document: machine_learning.txt | Score: 0.511

Question: Who is Batman?
No relevant text found.


##  5. Routing

### 5.1  Route definition

In [62]:
class Route(str, Enum):
    TEXT = "text"
    SUPERHERO = "superhero"
    BOTH = "both"
    UNKNOWN = "unknown"

##  5.2 Routing Decision

In [63]:
class RoutingDecision(BaseModel):
    route: Route
    superhero_name: Optional[str] = None
    reason: str = Field(min_length=1, max_length=300)

### 5.3 LLM-Based Router

In [64]:
ROUTER_SYSTEM_PROMPT = """
You are a routing component for a question-answering system.

The system has two information sources:

1. TEXT:
   A local text dataset containing domain-specific information.

2. SUPERHERO:
   The Superhero API, which contains information about superhero characters.

Your job is to determine which source or sources are needed to answer the user's question.

Choose exactly one route:

- "text": The question can be answered using the text dataset.
- "superhero": The question requires superhero information.
- "both": The question requires information from both sources.
- "unknown": Neither source is sufficient or the intent is unclear.

If the question refers to a superhero, identify the superhero name when possible.

Return ONLY valid JSON with this structure:

{
    "route": "text | superhero | both | unknown",
    "superhero_name": "string or null",
    "reason": "short explanation"
}
"""


def llm_route(question: str) -> RoutingDecision:
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": ROUTER_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": question
            }
        ],
        response_format={"type": "json_object"}
    )

    raw_output = response.choices[0].message.content

    decision = RoutingDecision.model_validate(
        json.loads(raw_output)
    )

    return decision

## 6. Deterministic Route Verification

### 6.1 verify_route()

In [65]:
def verify_route(question: str, decision: RoutingDecision):
    """
    Verify the LLM routing decision using the actual data sources.

    Returns:
        verified_decision: The verified routing decision.
        retrieved_text: Text retrieval results, if available.
        superhero_data: Superhero API result, if available.
    """

    retrieved_text = []
    superhero_data = None

    if decision.route == Route.TEXT:
        retrieved_text = retrieve_text(
            question,
            top_k=3,
            threshold=0.1
        )

        if retrieved_text:
            return decision, retrieved_text, superhero_data

        return (
            RoutingDecision(
                route=Route.UNKNOWN,
                superhero_name=None,
                reason="No sufficiently relevant text information was found."
            ),
            retrieved_text,
            superhero_data
        )

    if decision.route == Route.SUPERHERO:
        if not decision.superhero_name:
            return (
                RoutingDecision(
                    route=Route.UNKNOWN,
                    superhero_name=None,
                    reason="No superhero name was identified."
                ),
                retrieved_text,
                superhero_data
            )

        try:
            superhero_data = search_superhero(decision.superhero_name)

            if superhero_data.get("response") == "success":
                return decision, retrieved_text, superhero_data

        except RuntimeError:
            pass

        return (
            RoutingDecision(
                route=Route.UNKNOWN,
                superhero_name=decision.superhero_name,
                reason="The Superhero API did not return valid information."
            ),
            retrieved_text,
            superhero_data
        )

    if decision.route == Route.BOTH:
        retrieved_text = retrieve_text(
            question,
            top_k=3,
            threshold=0.1
        )

        if decision.superhero_name:
            try:
                superhero_data = search_superhero(
                    decision.superhero_name
                )

                if superhero_data.get("response") != "success":
                    superhero_data = None

            except RuntimeError:
                superhero_data = None

        text_available = bool(retrieved_text)
        superhero_available = superhero_data is not None

        if text_available and superhero_available:
            return decision, retrieved_text, superhero_data

        if text_available:
            return (
                RoutingDecision(
                    route=Route.TEXT,
                    superhero_name=decision.superhero_name,
                    reason="Text information was available, but superhero information was unavailable."
                ),
                retrieved_text,
                superhero_data
            )

        if superhero_available:
            return (
                RoutingDecision(
                    route=Route.SUPERHERO,
                    superhero_name=decision.superhero_name,
                    reason="Superhero information was available, but no relevant text information was found."
                ),
                retrieved_text,
                superhero_data
            )

        return (
            RoutingDecision(
                route=Route.UNKNOWN,
                superhero_name=decision.superhero_name,
                reason="Neither source provided usable information."
            ),
            retrieved_text,
            superhero_data
        )

    return decision, retrieved_text, superhero_data

### 6.2 hybrid_route()

In [66]:
def hybrid_route(question: str):
    """
    Route a question using the LLM and verify the decision
    using the actual available sources.
    """
    decision = llm_route(question)

    verified_decision, retrieved_text, superhero_data = verify_route(
        question,
        decision
    )

    return verified_decision, retrieved_text, superhero_data

### 6.3 Test Hybrid Routing

In [67]:
routing_test_questions = [
    "What is retrieval augmented generation?",
    "Who is Batman?",
    "What is RAG and could Batman use it?",
    "What is the capital of France?"
]

for question in routing_test_questions:

    decision, retrieved_text, superhero_data = hybrid_route(question)

    print("\nQuestion:", question)
    print("Route:", decision.route.value)
    print("Superhero:", decision.superhero_name)
    print("Reason:", decision.reason)


Question: What is retrieval augmented generation?
Route: text
Superhero: None
Reason: General NLP concept, not superhero-related

Question: Who is Batman?
Route: superhero
Superhero: Batman
Reason: Question asks for superhero identity

Question: What is RAG and could Batman use it?
Route: both
Superhero: Batman
Reason: RAG is a general NLP concept (text source) and the question about Batman’s ability to use it requires superhero info.

Question: What is the capital of France?
Route: unknown
Superhero: None
Reason: The question asks for general knowledge not covered by the superhero API or the domain-specific text dataset.


## 7. Source context

### 7.1 Superhero  context

In [68]:
def get_superhero_context(
    superhero_data: dict,
    superhero_name: str
):
    """
    Extract the most relevant superhero information
    from already retrieved API data.
    """

    if not superhero_data:
        return None

    results = superhero_data.get("results", [])

    if not results:
        return None

    # Prefer an exact name match when multiple results are returned
    for superhero in results:
        if superhero.get("name", "").lower() == superhero_name.lower():
            return superhero

    # Fall back to the first result if no exact match is found
    return results[0]

In [69]:
superhero_data = search_superhero("Batman")

superhero = get_superhero_context(
    superhero_data,
    "Batman"
)

print("Name:", superhero["name"])
print("Full name:", superhero["biography"]["full-name"])
print("Publisher:", superhero["biography"]["publisher"])
print("Powerstats:", superhero["powerstats"])

Name: Batman
Full name: Terry McGinnis
Publisher: DC Comics
Powerstats: {'intelligence': '81', 'strength': '40', 'speed': '29', 'durability': '55', 'power': '63', 'combat': '90'}


### 7.2 Text Context

In [70]:
def get_text_context(retrieved_text):
    """
    Use already retrieved text results as the context.
    """
    return retrieved_text

In [99]:
text_results = retrieve_text(
    "What is retrieval augmented generation?"
)

text_context = get_text_context(text_results)

for item in text_context:

    print("Source:", item["document"])
    print("Score:", round(item["score"], 3))
    print("Content:", item["content"][:300])
    print()

Source: llm.txt
Score: 0.293
Content: Large Language Models (LLMs) are machine learning models trained on large collections of text. They can perform tasks such as text generation, summarization, question answering, and information extraction. LLMs generate responses based on patterns learned during training. They can also be provided w

Source: rag.txt
Score: 0.241
Content: Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text generation. Instead of relying only on the knowledge stored in a language model, RAG retrieves relevant documents from an external knowledge source and provides them to the language model as context. A 



## 8. Answer Generation

 ### 8.1 Prompt

In [72]:
ANSWER_SYSTEM_PROMPT = """
You are a helpful question-answering assistant.

Answer the user's question using ONLY the information provided
in the source context.

Do not invent facts that are not present in the source context.

If the provided sources do not contain enough information to answer
the question, clearly say that the available sources are insufficient.

Every answer MUST include a "Sources" section that identifies where
the information came from.

For text sources, use the document filename.

For superhero information, identify the source as:
Superhero API.
"""

### 8.2 Generate an Answer

In [73]:
def generate_answer(
    question: str,
    source_context: list
) -> str:
    """
    Generate the final answer using the hosted LLM.
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": ANSWER_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": f"""
Question:
{question}

Source context:
{source_context}
"""
            }
        ]
    )

    return response.choices[0].message.content

### 8.3 Test the Answer Generator

In [74]:
test_context = """
Source: rag.txt

Retrieval-Augmented Generation (RAG) is a technique that combines
information retrieval with text generation. A RAG system retrieves
relevant documents from an external knowledge source and provides
them to the language model as context.
"""

answer = generate_answer(
    question="What is retrieval augmented generation?",
    source_context=test_context
)

print(answer)

Retrieval Augmented Generation (RAG) is a technique that combines information retrieval with text generation. A RAG system retrieves relevant documents from an external knowledge source and provides them to the language model as context.  

**Sources**  
- rag.txt



## 9. End-to-End Pipeline

### 9.1  Answer Question

In [75]:
def answer_question(question: str):
    """
    Route the question, reuse the verified retrieved data,
    and generate the final answer.
    """

    decision, retrieved_text, superhero_data = hybrid_route(question)

    if decision.route == Route.UNKNOWN:
        return {
            "answer": "I could not find enough relevant information in the available sources to answer this question.",
            "sources": []
        }

    source_context = []
    sources = []

    if decision.route in (Route.TEXT, Route.BOTH):

        text_context = get_text_context(retrieved_text)

        if text_context:
            source_context.append({
                "source": "Text Knowledge Base",
                "content": text_context
            })

            sources.extend([
                item["document"]
                for item in text_context
            ])

    if decision.route in (Route.SUPERHERO, Route.BOTH):

        superhero_context = get_superhero_context(
            superhero_data,
            decision.superhero_name
        )

        if superhero_context:
            source_context.append({
                "source": "Superhero API",
                "content": superhero_context
            })

            sources.append("Superhero API")

    answer = generate_answer(
        question,
        source_context
    )

    return {
        "answer": answer,
        "sources": list(dict.fromkeys(sources))
    }

### 9.2 End-to-End Tests

In [76]:
test_questions = [
    "What is retrieval augmented generation?",
    "Who is Batman?",
    "What powers does Superman have?",
    "What is RAG and could Batman use it?"
]

for question in test_questions:

    print("\n" + "=" * 80)
    print("QUESTION:", question)
    print("=" * 80)

    answer = answer_question(question)

    print(answer)


QUESTION: What is retrieval augmented generation?
{'answer': 'Retrieval Augmented Generation (RAG) is a technique that combines information retrieval with text generation. Instead of relying solely on the knowledge stored inside a language model, a RAG system first retrieves relevant documents or passages from an external knowledge source. These retrieved passages are then supplied to a language model, which uses them as context to generate an answer. This approach is especially useful when the required information comes from a private, changing, or domain‑specific knowledge base.  \n\n**Sources**  \n- rag.txt  \n- llm.txt', 'sources': ['llm.txt', 'rag.txt']}

QUESTION: Who is Batman?
{'answer': 'Batman is a superhero from DC Comics. According to the Superhero API, his full name is **Terry\u202fMcGinnis**. He is a human male, 5\'10" tall (178\u202fcm) and weighs 170\u202flb (77\u202fkg). He has blue eyes and black hair. Batman’s power statistics include an intelligence of 81, strength

## 10. FastAPI Application

In [77]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

### 10.1  Request model

In [78]:

class AskRequest(BaseModel):
    question: str = Field(
        description="Natural language question for the chatbot"
    )

### 10.2 Response model

In [79]:
class AskResponse(BaseModel):
    question: str
    answer: str
    sources: list[str]

### 10.3 the FastAPI application

In [80]:
app = FastAPI(
    title="AI Engineer Assessment Chatbot",
    description="Hybrid chatbot using a text dataset, Superhero API, and hosted LLM.",
    version="1.0.0"
)

### 10.4 Create / ask endpoint

In [81]:
@app.post("/ask", response_model=AskResponse)
def ask(request: AskRequest):

    question = request.question.strip()

    # Validate empty questions
    if not question:
        raise HTTPException(
            status_code=400,
            detail="Question cannot be empty."
        )

    # Validate question length
    if len(question) > 1000:
        raise HTTPException(
            status_code=400,
            detail="Question is too long. Maximum length is 1000 characters."
        )

    try:
        result = answer_question(question)

        return AskResponse(
            question=question,
            answer=result["answer"],
            sources=result["sources"]
        )

    except RuntimeError as e:
        raise HTTPException(
            status_code=502,
            detail=str(e)
        )

    except Exception:
        raise HTTPException(
            status_code=500,
            detail="An unexpected error occurred while processing the question."
        )

### 10.5 test the endpoint

### Rag test

In [82]:
from fastapi.testclient import TestClient

test_client = TestClient(app)

response = test_client.post(
    "/ask",
    json={
        "question": "What is retrieval augmented generation?"
    }
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

c:\Users\GCB\Downloads\engineer_test\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


Status code: 200
Response:
{'question': 'What is retrieval augmented generation?', 'answer': 'Retrieval Augmented Generation (RAG) is a technique that combines information retrieval with text generation. Instead of relying solely on the knowledge stored inside a language model, a RAG system first retrieves relevant documents or passages from an external knowledge source. These retrieved passages are then supplied to a language model, which uses them as context to generate an answer. This approach is especially useful when the required information comes from a private, changing, or domain‑specific knowledge base.  \n\n**Sources**  \n- rag.txt  \n- llm.txt', 'sources': ['llm.txt', 'rag.txt']}


### Batman test

In [83]:
response = test_client.post(
    "/ask",
    json={
        "question": "Who is Batman?"
    }
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

Status code: 200
Response:
{'question': 'Who is Batman?', 'answer': 'Batman is Terry\u202fMcGinnis, a human male from Gotham City in the 25th\u202fCentury. He is the protagonist of *Batman\u202fBeyond*, first appearing in *Batman\u202fBeyond #1*. He is aligned with the good side, works as a member of the Batman Family and the Justice League Unlimited, and is the son of Warren\u202fMcGinnis and Mary\u202fMcGinnis, with a brother named Matt\u202fMcGinnis.  \n\n**Sources**  \n- Superhero API (Batman entry)', 'sources': ['Superhero API']}


### Both test 

In [84]:
response = test_client.post(
    "/ask",
    json={
        "question": "What is RAG and could Batman use it?"
    }
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

Status code: 200
Response:
{'question': 'What is RAG and could Batman use it?', 'answer': '**What is RAG?**  \nRetrieval‑Augmented Generation (RAG) is a technique that combines information retrieval with text generation. Instead of relying only on the knowledge stored in a language model, RAG retrieves relevant documents from an external knowledge source and provides them to the language model as context. A typical RAG system has two main stages:  \n1. **Retriever** – searches a collection of documents and finds the most relevant passages.  \n2. **Generator** – uses the retrieved passages to generate an answer.  \nRAG is especially useful when the information comes from a private, changing, or domain‑specific knowledge base.  \n\n**Could Batman use RAG?**  \nThe source data gives Batman’s intelligence score as 81, indicating a high level of intellectual capability. While the information does not explicitly state whether Batman has used or could use RAG, his high intelligence suggests t

## 11. Error Handling Tests

The chatbot should handle invalid input and external service failures gracefully.

We test:
- Empty questions
- Questions longer than 1000 characters
- Superhero API failure
- Hosted LLM failure
- Unexpected application failure


### Test empty question

In [85]:

response = test_client.post(
    "/ask",
    json={
        "question": ""
    }
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

Status code: 400
Response:
{'detail': 'Question cannot be empty.'}


### Test whitespace-only question


In [100]:

response = test_client.post(
    "/ask",
    json={
        "question": "   "
    }
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

Status code: 400
Response:
{'detail': 'Question cannot be empty.'}


### Test missing question field


In [101]:

response = test_client.post(
    "/ask",
    json={}
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

Status code: 422
Response:
{'detail': [{'type': 'missing', 'loc': ['body', 'question'], 'msg': 'Field required', 'input': {}}]}


### Test a question longer than 1000 characters


In [86]:

long_question = "What is RAG? " * 100

response = test_client.post(
    "/ask",
    json={
        "question": long_question
    }
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

Status code: 400
Response:
{'detail': 'Question is too long. Maximum length is 1000 characters.'}


### Test invalid question type


In [102]:

response = test_client.post(
    "/ask",
    json={
        "question": 123
    }
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

Status code: 422
Response:
{'detail': [{'type': 'string_type', 'loc': ['body', 'question'], 'msg': 'Input should be a valid string', 'input': 123}]}


### Temporarily simulate a Superhero API failure


In [87]:

original_search_superhero = search_superhero

def failing_search_superhero(name: str) -> dict:
    raise RuntimeError("Simulated Superhero API failure")


search_superhero = failing_search_superhero

### Test how /ask handles the Superhero API failure


In [88]:

response = test_client.post(
    "/ask",
    json={
        "question": "Who is Batman?"
    }
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

Status code: 200
Response:
{'question': 'Who is Batman?', 'answer': 'I could not find enough relevant information in the available sources to answer this question.', 'sources': []}


### Restore the original Superhero API function


In [89]:
search_superhero = original_search_superhero

### Test hosted LLM failure

In [90]:
# Save the original LLM function
original_generate_answer = generate_answer

In [91]:
# Temporarily simulate a hosted LLM failure
def failing_generate_answer(
    question: str,
    source_context: str
) -> str:
    raise RuntimeError("Simulated LLM API failure")

In [92]:
generate_answer = failing_generate_answer

### Test how `/ask` handles a hosted LLM failure.

In [93]:
response = test_client.post(
    "/ask",
    json={"question": "What is RAG?"}
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

Status code: 502
Response:
{'detail': 'Simulated LLM API failure'}


### Restore the original LLM function

In [94]:
generate_answer = original_generate_answer

### Test unexpected application failure

In [95]:
original_answer_question = answer_question

def failing_answer_question(question: str) -> dict:
    raise ValueError("Simulated unexpected failure")

answer_question = failing_answer_question

### Test how `/ask` handles an unexpected application failure

In [96]:
response = test_client.post(
    "/ask",
    json={
        "question": "What is RAG?"
    }
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

Status code: 500
Response:
{'detail': 'An unexpected error occurred while processing the question.'}


In [97]:
answer_question = original_answer_question

### Test normal request after restoring all services

In [98]:
response = test_client.post(
    "/ask",
    json={
        "question": "What is retrieval augmented generation?"
    }
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

Status code: 200
Response:
{'question': 'What is retrieval augmented generation?', 'answer': 'Retrieval Augmented Generation (RAG) is a technique that combines information retrieval with text generation. Instead of relying solely on the knowledge stored inside a language model, a RAG system first retrieves relevant documents or passages from an external knowledge source. These retrieved passages are then supplied to a language model, which uses them as context to generate an answer. This approach is especially useful when the required information comes from a private, changing, or domain‑specific knowledge base.  \n\n**Sources**  \n- rag.txt  \n- llm.txt', 'sources': ['llm.txt', 'rag.txt']}
